## For positive and negative neuron controls within 80K NGN2 derived neurons
- 'C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK', 'C_positive_heart_MK', 'C_negative_heart_MK', 'C_positive_neuron_CD'

In [30]:
from importlib import reload
import pandas as pd
import sys
import os
import yaml

sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [31]:
# helpful functions

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'

def get_category(header):
    """Get the category of the headers set all to element if not scarmbled
    """

    label = hf.get_label(header)

    if 'scramble' in header:
        # Check if header is scrambled
        # Info: scrambled is in C_negative_neuron_NP and scramble MK
        # Note: Other cases are not checked with this function
        return 'scrambled'
    else:
        return 'element'


def get_reference_genome(row):
    """If col_category is synthetic or scrambled or in dnase_control_groups, set ref to GRCh37 else GRCh38"""
    row[col_ref] = 'GRCh38'
    return row

def extract_rsid(s):
    import re
    match = re.findall(r"rs\d+", s)
    return match[-1] if match else None

import requests

def get_rsid_position(rsid, assembly="GRCh38"):
    url = f"https://rest.ensembl.org/variation/human/{rsid}"
    headers = {"Content-Type": "application/json"}
    response = requests.get(url, headers=headers, params={"genome": assembly})
    if response.status_code == 200:
        data = response.json()
        for mapping in data.get("mappings", []):
            if mapping["assembly_name"] == assembly:
                return {
                    "chromosome": mapping["seq_region_name"],
                    "start": mapping["start"],
                    "end": mapping["end"],
                }
    return None




def get_start_end_strand_control(row):
    """
    Special cases to set chr, start, end and strand for control sequences from their header (because not in region bed)

    Case: C_negative_neuron_MK: (all C_negative_neuron_MK sequences have "_chr" pattern)
            header: C_negative_neuron_MK:tile_14444_chr15_67066278_67066547_reference__1.1385203581298
                => chr: 15, start: 67066278, end: 67066547, strand: . (no information given)
            variant_header: >C_negative_neuron_MK:tile_36043_chr6_14500968_14501237_G_C_261__0.274168942409752

    Case: C_positive_neuron_MK: (all C_positive_neuron_MK sequences have "_chr" pattern")
            header: C_positive_neuron_MK:tile_35742_chr6_3247831_3248100_reference_0.892141141777512
                => chr: 6, start: 3247831, end: 3248100, strand: . (no information given)

    Case: C_negative_heart_MK: ( all C_negative_heart_MK sequences have "_chr" pattern)
            header: C_negative_heart_MK:tile_6903_chr11_9614045_9614314_reference__0.958461950470297
                => chr: 11, start: 9614045, end: 9614314, strand: . (no information given)

    Case: C_positive_heart_MK: (all C_positive_heart_MK sequences have "_chr" pattern)
            header: C_positive_heart_MK:tile_7939_chr11_65487592_65487861_reference_1.25449216981846
                => chr: 11, start: 65487592, end: 65487861, strand: . (no information given)

    Case: C_negative_neuron_NP: (all C_negative_neuron_NP sequences have "_chr" pattern)
            header: C_negative_neuron_NP:GW18_PFC_ABC_chr15_89400286_89400556_0.830617698776558
                => chr: 15, start: 89400286, end: 89400556, strand: . (no information given)
            additional condition: scrambled_control____2.28928058620308
                => category: scrambled

    Case: C_positive_neuron_NP: (all C_positive_neuron_NP sequences have "_chr" pattern)
            header: C_positive_neuron_NP:GW18_PFC_ABC_chr11_65487667_65487937_5.27702983385667
                => chr: 11, start: 65487667, end: 65487937, strand: . (no information given)

    Case: C_positive_neuron_CD: headers have "::chr" pattern and delimited by "-mean_ratio"
            header: C_positive_neuron_CD:p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42
                => chr: 4, start: 155359187, end: 155359457, strand: . (no information given)
            additional condition: C_positive_neuron_CD:c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31

    """

    if row[col_category] == 'scrambled':
        row[col_class] = "element inactive control"
        return row
    name = row[col_name]

    if name.startswith('C_negative_neuron_MK') or name.startswith('C_positive_neuron_MK'):
        row[col_source] = 'Michael Kosicki'
        row[col_class] = 'element inactive control'
        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row

        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        row[col_category] = 'element'
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1]) -1 # turn into 0-based
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '+'
        if len(name.split('_chr')[1].split("_")) == 7:
            row[col_variant_class] = 'SNV'
            row[col_category] = 'variant'
            row[col_class] = 'variant negative control'
            row[my_col_ref_base] = name.split('_chr')[1].split("_")[3]
            row[my_col_alt_base] = name.split('_chr')[1].split("_")[4]
            row[col_variant_pos] = int(name.split('_chr')[1].split("_")[5]) - 1
            row[col_allele] = 'alt'
            if 'positive_neuron' in name:
                row[col_class] = 'variant positive control'

    elif name.startswith('C_negative_neuron_NP') or name.startswith('C_positive_neuron_NP'):
        row[col_class] = 'element inactive control'
        row[col_source] = 'Nick Page'

        if 'scramble' in name:
            row[col_category] = 'scrambled'
            return row
        if 'positive_neuron' in name:
            row[col_class] = 'element active control'
        region_info = '_'.join(name.split('_chr')[1].split('_')[:3]) # 11_9614045_9614314
        # print(region_info)
        row[col_chr] = f"chr{region_info.split('_')[0]}"
        row[col_start] = int(region_info.split('_')[1])
        row[col_end] = int(region_info.split('_')[2])
        row[col_strand] = '+'

    elif name.startswith('C_positive_neuron_CD'):
        # note the given positions of the variants are given as 50% but sometimes it is n/2 and sometimes n/2+1 => rsid to position
        # used the explanation from Chengyu mail: 10.12.2024
        row[col_class] = 'element inactive control'
        row[col_source] = 'Chengyu Deng'

        if 'NA_NA_NA' in name:
            row[col_category] = 'scrambled'
            row[col_chr] = 'NA'
            row[col_start] = 'NA'
            row[col_end] = 'NA'
            row[col_strand] = 'NA'
            return row
        row[col_class] = 'element active control'
        region_info = name.split('::chr')[1].split('-mean_ratio')[0] # 4:155359187-155359457
        row[col_chr] = f"chr{region_info.split(':')[0]}"
        row[col_start] = int(region_info.split(':')[1].split('-')[0])
        row[col_end] = int(region_info.split(':')[1].split('-')[1])
        row[col_strand] = '+'

        variant_info = name.split('C_positive_neuron_CD:')[1].split('::chr')[0].split('_')
        rsid = extract_rsid(name) # e.g. p1_rs10061048_A_G_ref_50_A::chr5:1030722-1030992-mean_ratio2.09
        position = get_rsid_position(rsid)
        if position:
            row[col_variant_class] = 'SNV'
            row[col_category] = 'variant'
            row[col_class] = 'variant positive control'
            row[col_allele] = 'alt'
            row[my_col_ref_base] = variant_info[2]
            row[my_col_alt_base] = variant_info[3]
            row[col_variant_pos] = int(position['start']) - 1 - row[col_start] # 1-based start (=> 0-based location should be start - 1)
            if '_ref_' in name:
                row[col_allele] = 'ref'
                reference_info = 'possible reference but no alternative assigned'
                row[col_variant_pos] = 'NA'
                row[my_col_alt_base] = 'NA'
                row[my_col_ref_base] = 'NA'

        else:
            print('Rsid not found: ', rsid)
    return row


# dict of chr number to refseq chromosome number
chrom_2_refseq = {
    "chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) # I think everything is now 0-based
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'


def get_spdi(row):
    """
    Returns the SPDI identifier for the given variant using start and variant_pos
    Assumption: allele need to be set beforehands
    """
    if not (hf.is_alternative(row[col_allele])) and (row[my_col_ref_base] and row[my_col_alt_base]):
        row['SPDI'] = 'NA'
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = f'{row[col_chr]}-{row[col_start]+row[col_variant_pos]}-{row[my_col_ref_base]}-{row[my_col_alt_base]}'
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row[col_SPDI] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row


In [32]:
input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
pre_metadata_df

# split the metadata file headers by '#'
# Apply the function to each row and concatenate the results
pre_metadata_df_split = pd.concat(pre_metadata_df.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# Reset the index
pre_metadata_df_split.reset_index(drop=True, inplace=True)

pre_metadata_df = pre_metadata_df_split.copy()
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))
print(pre_metadata_df.shape[0]) # 80803

80803


In [33]:
underscore_parsable_headers = ['C_negative_neuron_NP', 'C_positive_neuron_NP', 'C_positive_neuron_MK',  'C_negative_neuron_MK', 'C_positive_neuron_CD']

# # filter for underscore parsable headers
pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'].isin(underscore_parsable_headers)].copy()
print(pre_metadata_df_filtered.shape[0]) # 971 splitting duplicates: 1107

# add the columns of the metadata file
pre_metadata_df_filtered[col_category] = 'NA'
pre_metadata_df_filtered[col_class] = 'NA'
pre_metadata_df_filtered[col_source] = 'NA'
pre_metadata_df_filtered[col_ref] = 'NA'
pre_metadata_df_filtered[col_chr] = 'NA'
pre_metadata_df_filtered[col_start] = 'NA'
pre_metadata_df_filtered[col_end] = 'NA'
pre_metadata_df_filtered[col_strand] = 'NA'
pre_metadata_df_filtered[col_variant_class] = 'NA'
pre_metadata_df_filtered[col_variant_pos] = 'NA'
pre_metadata_df_filtered[col_SPDI] = 'NA'
pre_metadata_df_filtered[col_allele] = 'NA'
pre_metadata_df_filtered[col_info] = ''


pre_metadata_df_filtered[col_category] = pre_metadata_df_filtered[col_name].apply(get_category)
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_reference_genome, axis=1)

# parse regions from header
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(get_start_end_strand_control, axis=1)
pre_metadata_df_filtered

# add SPDI
pre_metadata_df_filtered = pre_metadata_df_filtered.apply(lambda row: get_spdi(row), axis=1)

# remove adapter from sequence (15bp of start and end):
pre_metadata_df_filtered[col_sequence] = pre_metadata_df_filtered[col_sequence].apply(lambda x: x[15:-15])

746


In [34]:
pre_metadata_df_filtered.loc[pre_metadata_df_filtered[col_class] == "NA"]

,SPDI,allele,category,chr,class,end,info,name,ref,sequence,source,start,strand,tmp_alt_base,tmp_label,tmp_ref_base,variant_class,variant_pos


In [35]:
interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]
group_name = 'neuro_controls'
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
for group_name in underscore_parsable_headers:
    pre_metadata_df_filtered_group = pre_metadata_df_filtered.loc[pre_metadata_df_filtered['tmp_label'] == group_name].copy()
    pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()
    print(group_name)
    print(f'Expected: {pre_metadata_df_group.shape[0]}', )
    print(f'Actuall number: {pre_metadata_df_filtered_group.shape[0]}')
    # show the duplicated sequences
    print(f'Unique header number: {pre_metadata_df_filtered_group[col_sequence].nunique()}')
    # Write DataFrame to TSV file
    pre_metadata_df_filtered_group[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
    os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')


C_negative_neuron_NP
Expected: 217
Actuall number: 217
Unique header number: 217
C_positive_neuron_NP
Expected: 99
Actuall number: 99
Unique header number: 99
C_positive_neuron_MK
Expected: 100
Actuall number: 100
Unique header number: 100
C_negative_neuron_MK
Expected: 234
Actuall number: 234
Unique header number: 234
C_positive_neuron_CD
Expected: 96
Actuall number: 96
Unique header number: 96


In [36]:
# not useful
# group_name = 'neuro_controls'
# output_dir = config['final_output_dir']
# output_path = os.path.join(output_dir, group_name)
# # Write DataFrame to TSV file
# pre_metadata_df_filtered[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
# os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')
# # Write DataFrame to TSV file
# pre_metadata_df_filtered[interesting_columns].to_csv('/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tmp.tsv.gz', sep='\t', index=False, na_rep='NA', compression='gzip')
# import os
# os.system('zcat /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > /data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/neuro_controls/neuro_controls.metadata.tsv.gz')

In [37]:
breaking spot

SyntaxError: invalid syntax (2270176476.py, line 1)

### Working on C_positive_neuron_CD (errors in the header derived SPDIs)

In [7]:
chengyu_names_df = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == 'C_positive_neuron_CD'].copy()
chengyu_names_df['no_label'] = chengyu_names_df[col_name].apply(lambda name: name.replace('C_positive_neuron_CD:', ''))
# chengyu_names_df[col_sequence] = chengyu_names_df[col_sequence].apply(lambda x: x[15:-15])
# chengyu_names_df[col_sequence].to_list()
chengyu_names_df

chengyu_names_df['no_label'].to_list()
# hf.write_fasta(chengyu_names_df, '80K_MPRA_group_C_positive_neuron_CD.fa', header=['no_label', 'sequence'])
# rs55985730: from dbsnp: NC_000007.14:g.128776990T>G 128776855 + 135 = 128776990
# rs9975055: from dbsnp: NC_000021.9:44930103:T:G 44929969 + 135 = 44930104

['c1_NA_NA_NA_NA::72hr_top_99-mean_ratio1.92',
 'c1_NA_NA_NA_NA::72hr_top_4-mean_ratio3.08',
 'p1_rs6813360_A_C_ref_50_A::chr4:155359187-155359457-mean_ratio2.42',
 'p1_rs55985730_T_G_alt_50_T::chr7:128776855-128777125-mean_ratio2.34',
 'c1_NA_NA_NA_NA::72hr_top_94-mean_ratio2.31',
 'p1_rs7115714_G_A_ref_50_A::chr11:120424017-120424287-mean_ratio2.22',
 'p1_rs9975055_T_G_alt_50_G::chr21:44929969-44930239-mean_ratio2.18',
 'n1_rs2279982_G_A_alt_50::chr2:164841902-164842172-mean_ratio2.16',
 'p1_rs34761481_G_A_alt_50_G::chr7:129161739-129162009-mean_ratio2.14',
 'p1_rs7115714_G_A_alt_50_A::chr11:120424017-120424287-mean_ratio2.13',
 'n1_rs275835_G_A_alt_50::chr7:132509137-132509407-mean_ratio2.11',
 'p1_rs10061048_A_G_ref_50_A::chr5:1030722-1030992-mean_ratio2.09',
 'p1_rs114772924_G_A_ref_50_A::chr1:32253766-32254036-mean_ratio2.08',
 'p1_rs11757302_C_T_ref_50_C::chr6:905837-906107-mean_ratio2.06',
 'p1_rs115202710_C_T_alt_50_C::chr6:24704205-24704475-mean_ratio2.05',
 'p1_rs62086577_G_

cases I tested